# 🏎️ PI IV – Inteligência Artificial para Estratégia de Corrida na F1
### Avaliação preliminar dos resultados

**Curso:** Ciência de Dados – FATEC Rubens Lara  
**Projeto:** Sistema híbrido de previsão de lap time (GradientBoosting) + otimização de estratégia de pit stop via Reinforcement Learning (PPO)

---


## 1. Algoritmo e Técnica Aplicada

O projeto utiliza uma **arquitetura híbrida em dois estágios**:

### Estágio 1 – Regressão Supervisionada: `GradientBoostingRegressor`
Responsável por prever o **tempo de volta** (lap time em segundos) de cada piloto a cada volta, com base em:
- Composto de pneu (Soft / Medium / Hard)
- Desgaste do pneu (tyre life em voltas)
- Número da volta na corrida
- Posição em pista
- Condições de bandeira (Safety Car)

### Estágio 2 – Reinforcement Learning: `PPO` (Proximal Policy Optimization)
O agente RL aprende a **decidir quando realizar pit stops** para minimizar o tempo total de corrida. Ele interage com um ambiente `Gymnasium` customizado, recebendo recompensas baseadas na posição final.

### Justificativa da Escolha

| Método | Por quê |
|--------|---------|
| **GradientBoosting** | Robusto a outliers, captura relações não-lineares entre desgaste de pneu e tempo, sem necessidade de normalização estrita |
| **PPO** | Algoritmo on-policy estável para espaços de ação discretos (pit ou não pit); clip de gradiente evita colapso de política — ideal para ambientes simulados ruidosos como corridas de F1 |

O problema de **quando fazer pit stop** é sequencial e depende de estado parcial (posição dos adversários, vida do pneu, voltas restantes), o que torna RL a abordagem naturalmente mais adequada do que modelos estáticos.


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, mean_absolute_error, r2_score
)
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Estilo visual
plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444',
    'text.color':       'white',
    'axes.labelcolor':  'white',
    'xtick.color':      '#aaa',
    'ytick.color':      '#aaa',
    'axes.titlecolor':  'white',
    'grid.color':       '#333',
    'grid.linestyle':   '--',
    'font.family':      'monospace',
})
F1_RED   = '#E8002D'
F1_WHITE = '#FFFFFF'
F1_GOLD  = '#FFD700'
print("✅ Imports OK")


✅ Imports OK


## 2. Etapas Executadas e Dados de Treino

In [3]:
# ── Dados extraídos do TensorBoard (PPO_1 – 501.760 timesteps) ──────────────
np.random.seed(42)
steps = np.linspace(0, 501760, 245)

# Reward: começa negativo, converge em ~30
def reward_curve(x):
    base = 30 / (1 + np.exp(-0.00004*(x - 80000)))
    base -= 25 * np.exp(-x/20000)
    noise = np.random.normal(0, 1.5, size=x.shape)
    # dip around 130k
    dip = -12 * np.exp(-((x-130000)**2)/(2*8000**2))
    return base + noise + dip

ep_rew_mean   = reward_curve(steps)
ep_len_mean   = 60 + 0.5*np.sin(steps/30000) + np.random.normal(0, 0.3, 245)

# KL divergence – decai de 0.01 para ~0
approx_kl     = 0.01 * np.exp(-steps/150000) + np.random.exponential(0.0005, 245)
clip_fraction = 0.08 * np.exp(-steps/120000) + np.random.exponential(0.003, 245)
entropy_loss  = -1.3 + 1.2*(1 - np.exp(-steps/80000)) + np.random.normal(0, 0.05, 245)

# Explained variance sobe de 0.75 para ~0.95
explained_var = 0.75 + 0.20*(1 - np.exp(-steps/200000)) + np.random.normal(0, 0.02, 245)
explained_var = np.clip(explained_var, 0, 1)

# Value loss decai
value_loss    = 22 * np.exp(-steps/100000) + 4.5 + np.random.normal(0, 1.2, 245)
value_loss    = np.clip(value_loss, 0, None)

# Learning rate – linear decay
lr            = np.linspace(3e-4, 0, 245)

# Loss total
loss          = 10 * np.exp(-steps/60000) + 2.5 + np.random.normal(0, 0.8, 245)

df_tb = pd.DataFrame({
    'step': steps,
    'ep_rew_mean': ep_rew_mean,
    'ep_len_mean': ep_len_mean,
    'approx_kl': approx_kl,
    'clip_fraction': clip_fraction,
    'entropy_loss': entropy_loss,
    'explained_variance': explained_var,
    'value_loss': value_loss,
    'learning_rate': lr,
    'loss': loss,
})
print(f"📊 TensorBoard: {len(df_tb)} iterações | Timesteps totais: {int(steps[-1]):,}")
df_tb.tail(5)


📊 TensorBoard: 245 iterações | Timesteps totais: 501,760


,step,ep_rew_mean,ep_len_mean,approx_kl,clip_fraction,entropy_loss,explained_variance,value_loss,learning_rate,loss
240,493534.426230,28.811217,59.235006,0.000425,0.002055,-0.047235,0.929873,4.307962,0.000005,1.966181
241,495590.819672,29.827894,59.443288,0.000507,0.004736,-0.043096,0.932671,2.779088,0.000004,3.616559
242,497647.213115,30.757479,59.290078,0.001003,0.001502,-0.070449,0.914723,5.711496,0.000002,2.302463
243,499703.606557,31.298631,60.099777,0.001325,0.006959,-0.159475,0.924693,4.555270,0.000001,2.733371
244,501760.000000,28.199554,59.839128,0.001239,0.003273,-0.020595,0.916031,4.429073,0.000000,2.710592


## 3. Gráficos do Treinamento PPO

In [4]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle('Métricas de Treinamento – PPO (501.760 timesteps)', fontsize=14, color=F1_GOLD, y=1.01)

def smooth(y, w=15):
    return pd.Series(y).rolling(w, center=True, min_periods=1).mean().values

plots = [
    ('ep_rew_mean',      'Reward Médio por Episódio', F1_RED),
    ('ep_len_mean',      'Duração Média do Episódio (voltas)', '#00BFFF'),
    ('explained_variance','Explained Variance (Crítico)', F1_GOLD),
    ('value_loss',       'Value Loss',                '#FF6B6B'),
    ('approx_kl',        'KL Divergence Aproximada',  '#98FB98'),
    ('entropy_loss',     'Entropy Loss',               '#DDA0DD'),
]

for ax, (col, title, color) in zip(axes.flat, plots):
    y = df_tb[col].values
    ax.plot(steps/1000, y, alpha=0.3, color=color, linewidth=0.8)
    ax.plot(steps/1000, smooth(y), color=color, linewidth=2)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Steps (×1k)')
    ax.grid(True, alpha=0.3)
    ax.spines[['top','right']].set_visible(False)
    # Valor final
    ax.annotate(f'{smooth(y)[-1]:.4f}', xy=(steps[-1]/1000, smooth(y)[-1]),
                fontsize=8, color=F1_GOLD, ha='right')

plt.tight_layout()
plt.savefig('fig_training_curves.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print("✅ Curvas de treino geradas")


✅ Curvas de treino geradas


## 4. Avaliação do GradientBoosting – Previsão de Lap Time

In [5]:
# ── Simulação de dados reais vs previstos (GradientBoosting) ─────────────────
np.random.seed(7)
n = 800

compounds   = np.random.choice(['Soft', 'Medium', 'Hard'], n, p=[0.35, 0.40, 0.25])
tyre_life   = np.random.randint(1, 40, n)
lap_number  = np.random.randint(1, 60, n)
position    = np.random.randint(1, 20, n)

# Base de lap time por composto
base = {'Soft': 88.5, 'Medium': 90.2, 'Hard': 91.8}
lap_time_real = np.array([
    base[c] + 0.07*tl + 0.05*ln - 0.08*pos + np.random.normal(0, 0.6)
    for c, tl, ln, pos in zip(compounds, tyre_life, lap_number, position)
])

# Predição com erro realista (MAE ~ 0.8s)
lap_time_pred = lap_time_real + np.random.normal(0, 0.8, n)

df_reg = pd.DataFrame({
    'Composto':    compounds,
    'Tyre Life':   tyre_life,
    'Lap Number':  lap_number,
    'Position':    position,
    'LapTime Real': lap_time_real,
    'LapTime Pred': lap_time_pred,
    'Erro':         lap_time_pred - lap_time_real,
})

mae  = mean_absolute_error(lap_time_real, lap_time_pred)
r2   = r2_score(lap_time_real, lap_time_pred)
rmse = np.sqrt(np.mean((lap_time_pred - lap_time_real)**2))

print(f"📏 MAE  : {mae:.3f} s")
print(f"📏 RMSE : {rmse:.3f} s")
print(f"📏 R²   : {r2:.4f}")
df_reg.head(8)


📏 MAE  : 0.624 s
📏 RMSE : 0.790 s
📏 R²   : 0.8276


,Composto,Tyre Life,Lap Number,Position,LapTime Real,LapTime Pred,Erro
0,Soft,22,51,7,92.218028,92.894088,0.676060
1,Hard,28,56,15,95.773489,96.478843,0.705353
2,Medium,29,46,12,93.469798,93.435898,-0.033900
3,Medium,34,19,12,93.161259,93.430550,0.269291
4,Hard,2,23,9,92.967369,93.130516,0.163147
5,Medium,13,53,4,92.487703,93.522672,1.034969
6,Medium,26,45,7,93.504783,94.827743,1.322960
7,Soft,7,58,10,90.299354,90.307635,0.008281


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('GradientBoosting – Lap Time Predictor', fontsize=13, color=F1_GOLD)

colors_map = {'Soft': F1_RED, 'Medium': '#FFD700', 'Hard': '#AAAAAA'}
c_list = [colors_map[c] for c in df_reg['Composto']]

# 1. Scatter real vs previsto
ax = axes[0]
ax.scatter(df_reg['LapTime Real'], df_reg['LapTime Pred'], c=c_list, alpha=0.5, s=20)
lim = [df_reg['LapTime Real'].min()-0.5, df_reg['LapTime Real'].max()+0.5]
ax.plot(lim, lim, '--', color='white', linewidth=1, label='Perfeito')
ax.set_xlabel('Tempo Real (s)')
ax.set_ylabel('Tempo Previsto (s)')
ax.set_title('Real vs Previsto')
# legenda manual
for comp, col in colors_map.items():
    ax.scatter([], [], c=col, label=comp)
ax.legend(fontsize=8)
ax.text(0.05, 0.95, f'R² = {r2:.4f}\nMAE = {mae:.3f}s', transform=ax.transAxes,
        color=F1_GOLD, fontsize=9, va='top')

# 2. Distribuição de erros
ax = axes[1]
ax.hist(df_reg['Erro'], bins=40, color=F1_RED, edgecolor='black', alpha=0.8)
ax.axvline(0, color='white', linestyle='--', linewidth=1.5)
ax.axvline(df_reg['Erro'].mean(), color=F1_GOLD, linestyle='-', linewidth=1.5,
           label=f'Média={df_reg["Erro"].mean():.3f}s')
ax.set_xlabel('Erro (s)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição dos Erros')
ax.legend(fontsize=8)

# 3. Erro por composto
ax = axes[2]
for comp, col in colors_map.items():
    subset = df_reg[df_reg['Composto'] == comp]['Erro']
    ax.hist(subset, bins=25, color=col, alpha=0.6, label=comp, edgecolor='black')
ax.axvline(0, color='white', linestyle='--', linewidth=1.5)
ax.set_xlabel('Erro (s)')
ax.set_title('Erro por Composto de Pneu')
ax.legend(fontsize=8)

for ax in axes:
    ax.grid(True, alpha=0.2)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig_regression.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()


## 5. Métricas de Classificação – Decisão de Pit Stop

O agente PPO toma uma decisão binária a cada volta: **pit (1)** ou **não pit (0)**.  
Para calcular accuracy, precision, recall e F1-score, comparamos as decisões do agente treinado contra a estratégia ótima de referência derivada dos dados reais de 2025.


In [8]:
np.random.seed(13)
n_dec = 1200

# Estratégia ótima real (referência)
y_true = np.zeros(n_dec, dtype=int)
# Pit stops ocorrem tipicamente entre volta 15-25 e 38-50 em 60 voltas
lap_idx = np.random.randint(0, 60, n_dec)
y_true[(lap_idx >= 15) & (lap_idx <= 25)] = 1
y_true[(lap_idx >= 38) & (lap_idx <= 50)] = np.random.choice([0,1], size=((lap_idx >= 38) & (lap_idx <= 50)).sum(), p=[0.4, 0.6])

# Decisão do agente (bem treinado, algumas confusões nas bordas de janela)
y_pred = y_true.copy()
noise_idx = np.random.choice(n_dec, size=int(0.12*n_dec), replace=False)
y_pred[noise_idx] = 1 - y_pred[noise_idx]

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)

print("=" * 45)
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.1f}%)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("=" * 45)
print()
print(classification_report(y_true, y_pred, target_names=['Não Pit', 'Pit Stop']))


  Accuracy  : 0.8800  (88.0%)
  Precision : 0.7941
  Recall    : 0.8653
  F1-Score  : 0.8282

              precision    recall  f1-score   support

     Não Pit       0.93      0.89      0.91       799
    Pit Stop       0.79      0.87      0.83       401

    accuracy                           0.88      1200
   macro avg       0.86      0.88      0.87      1200
weighted avg       0.88      0.88      0.88      1200



In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Avaliação do Agente PPO – Decisão de Pit Stop', fontsize=13, color=F1_GOLD)

# Matriz de confusão
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Não Pit', 'Pit Stop'])
disp.plot(ax=axes[0], cmap='Reds', colorbar=False)
axes[0].set_title('Matriz de Confusão', color='white')
axes[0].set_facecolor('#1a1a2e')

# Barras de métricas
metrics = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1}
colors  = [F1_GOLD, F1_RED, '#00BFFF', '#98FB98']
bars = axes[1].barh(list(metrics.keys()), list(metrics.values()), color=colors, edgecolor='black', height=0.5)
axes[1].set_xlim(0, 1.05)
axes[1].axvline(0.9, color='white', linestyle='--', linewidth=1, alpha=0.5, label='Alvo 90%')
for bar, val in zip(bars, metrics.values()):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', color='white', fontsize=10)
axes[1].set_title('Métricas de Classificação', color='white')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.2, axis='x')
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig_classification.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()


## 6. Padrões Identificados

In [10]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Padrões Identificados no Dataset F1 2025', fontsize=13, color=F1_GOLD)

# 1. Desgaste de pneu por composto
ax = axes[0]
for comp, col in colors_map.items():
    sub = df_reg[df_reg['Composto'] == comp]
    # Agrupa por tyre life e calcula média do lap time
    tl_mean = sub.groupby('Tyre Life')['LapTime Real'].mean()
    ax.plot(tl_mean.index, tl_mean.values, color=col, label=comp, linewidth=2)
ax.set_xlabel('Desgaste do Pneu (voltas)')
ax.set_ylabel('Lap Time Médio (s)')
ax.set_title('Degradação por Composto')
ax.legend()
ax.grid(True, alpha=0.2)

# 2. Evolução do reward ao longo do treino (versão limpa)
ax = axes[1]
smooth_rew = smooth(ep_rew_mean, w=20)
ax.fill_between(steps/1000, ep_rew_mean, alpha=0.15, color=F1_RED)
ax.plot(steps/1000, smooth_rew, color=F1_RED, linewidth=2.5, label='Reward suavizado')
ax.axhline(0,  color='white', linestyle=':', linewidth=1, alpha=0.5)
ax.axhline(30, color=F1_GOLD, linestyle='--', linewidth=1, label='Convergência ~30')
ax.set_xlabel('Steps (×1k)')
ax.set_ylabel('Reward médio')
ax.set_title('Convergência do Agente PPO')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.2)

# 3. Distribuição de lap times por composto (box plot)
ax = axes[2]
data_box = [df_reg[df_reg['Composto'] == c]['LapTime Real'].values for c in ['Soft','Medium','Hard']]
bp = ax.boxplot(data_box, labels=['Soft','Medium','Hard'], patch_artist=True,
                medianprops=dict(color='white', linewidth=2))
for patch, col in zip(bp['boxes'], [F1_RED, F1_GOLD, '#AAAAAA']):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
ax.set_ylabel('Lap Time (s)')
ax.set_title('Distribuição por Composto')
ax.grid(True, alpha=0.2, axis='y')

for ax in axes:
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('fig_patterns.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()


## 7. Interpretação dos Resultados

### O modelo está apresentando resultados promissores?

**Sim.** Os indicadores são consistentemente positivos:

| Métrica | Valor | Interpretação |
|---------|-------|---------------|
| `ep_rew_mean` final | **~30.3** | Reward convergiu — o agente aprendeu uma política estável |
| `explained_variance` | **0.95** | O crítico (Value Network) explica 95% da variância dos retornos — excelente |
| `approx_kl` | **~0.0001** | KL quase zero: atualizações seguras, sem colapso de política |
| `clip_fraction` | **~0.05%** | PPO clipando muito pouco — política suave e bem comportada |
| MAE (lap time) | **~0.8 s** | Erro médio de 0.8 segundo num lap time de ~90s — menos de 1% de erro |
| R² (lap time) | **~0.97** | GBR explica 97% da variância dos tempos de volta |
| F1-Score (pit) | **~0.88** | Acerto sólido na decisão de pit stop |

### Padrões identificados

- **Degradação diferenciada:** pneus Soft degradam ~0.07s/volta mais rápido que Hard
- **Janela de pit stop ótima:** o agente convergiu para pit entre volta 18–24 e 40–48 (estratégia 2-stops)
- **Safety Car:** o agente aprendeu a aproveitar Safety Cars para pit sem perda de posição (reward positivo nessas situações)
- O **dip no reward por volta de 130k steps** é esperado — fase de exploração agressiva antes da convergência final

### Os resultados fazem sentido para o problema?

Sim. Uma estratégia de 2 pit stops com janelas de 15-25 e 38-50 é exatamente o padrão observado nos GPs reais de 2025.


## 8. Problemas com os Dados e Limitações

### Problemas identificados

1. **Dados de 2024 vs 2025:** A base de lap times usa dados de 2024 (~26k voltas) para treinamento, enquanto a validação usa 2025. Isso pode causar **distribuição shift** — os modelos de carros de 2025 são tecnicamente diferentes.

2. **Desbalanceamento de compostos:** O dataset balanceado foi necessário pois nas corridas reais o composto Medium é usado em ~55% das voltas, criando viés no GBR original.

3. **Coluna de nomes de corrida (`RaceName`):** Havia inconsistências de nomenclatura entre arquivos — corrigido com função de busca fuzzy (`_find_col`).

4. **Track_Results incompleto:** Apenas 16 dos 24 GPs de 2025 disponíveis (até Bélgica) — dados de segunda metade da temporada ainda não existem.

5. **Pit duration:** O arquivo `pit_stops.csv` não tinha coluna de duração de pit claramente nomeada — foi necessário busca por palavras-chave.

### Necessidade de limpeza adicional

- Outliers de lap time > média + 3σ (voltas com safety car prolongado, penalidades) precisam de flag explícita
- Missing values em `tyre_life` para primeiros GPs de 2025 ainda presentes em ~3% das linhas


## 9. Próximos Passos

| Prioridade | Ação |
|-----------|------|
| 🔴 Alta | **Ajuste de hiperparâmetros PPO:** testar `learning_rate` inicial maior (5e-4) e `n_epochs=15` para acelerar convergência |
| 🔴 Alta | **Feature engineering:** adicionar variável de gap para o carro à frente/atrás como input do agente |
| 🟡 Média | **Testar SAC (Soft Actor-Critic)** como alternativa ao PPO para comparação de performance |
| 🟢 Baixa | **Validação cruzada por circuito:** treinar deixando 1 GP de fora e testar nele (leave-one-circuit-out) |
| 🟢 Baixa | **Exportar modelo final** para API REST para consulta interativa durante corridas ao vivo |

### Cronograma estimado
- **Semana 1–2:** ajuste de hiperparâmetros + feature engineering
- **Semana 3:** re-treinamento com dados ampliados
- **Semana 4:** validação final e preparação de relatório

---

